In [15]:
import os
import sys
from pathlib import Path
import json
import shutil
import subprocess
import warnings
import numpy as np
import yaml


NOTEBOOK_PATH = Path.cwd() 
if NOTEBOOK_PATH.name == "src":
    PROJECT_ROOT = NOTEBOOK_PATH.parent.resolve()
else:
    # If you launch the notebook from project root, ensure it still works:
    # if cwd contains 'src' directory, use cwd, else try parent
    if (NOTEBOOK_PATH / "src").exists():
        PROJECT_ROOT = NOTEBOOK_PATH.resolve()
    elif (NOTEBOOK_PATH.parent / "src").exists():
        PROJECT_ROOT = NOTEBOOK_PATH.parent.resolve()
    else:
        # fallback: assume cwd is project root
        PROJECT_ROOT = NOTEBOOK_PATH.resolve()

SRC = (PROJECT_ROOT / "src").resolve()

# Insert project root so "import src.xxx" works and relative imports inside src modules resolve
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path[0]:", sys.path[0])
print("Using src folder:", SRC)

# Now import from the src package (keeps relative imports inside src modules working)
from src.dataset_wrapper import find_and_load_datasets
from src.classifier_wrapper import SKLearnClassifierWrapper, RiverClassifierWrapper
from src.preprocessing_wrapper import PreprocessingWrapper
from src.logger import Logger
from src.features import FeatureExtraction

# sklearn helpers
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

# autoreload and warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


Project root added to sys.path[0]: /home/svobojan/StratosphereLinuxIPS/modules/flowmldetection/pipeline_ml_training
Using src folder: /home/svobojan/StratosphereLinuxIPS/modules/flowmldetection/pipeline_ml_training/src
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# paths, constants
root = "../../../../dataset-private/" # path to the datasets 
log_dir = "../../experiments/" # top level log directory
validation_split=0.1
experiment_name = "test_test_pipeline"
seed = 1111

EXPERIMENT_ROOT = Path("../experiments") / experiment_name
models_dir = EXPERIMENT_ROOT / "models"
preproc_dir = EXPERIMENT_ROOT / "preprocessing"
logs_dir = EXPERIMENT_ROOT / "logs"
results_dir = EXPERIMENT_ROOT / "results"
config_path = EXPERIMENT_ROOT / "config.txt"

if EXPERIMENT_ROOT.exists():
    shutil.rmtree(EXPERIMENT_ROOT)
EXPERIMENT_ROOT.mkdir(parents=True)

for d in [models_dir, preproc_dir, logs_dir, results_dir]:
    d.mkdir()


## Loading, init

In [17]:
loaders = find_and_load_datasets(root,cache_dir= "../cache/") #helper function from dataset_loader.py

found_datasets=list(loaders.keys())
print("Found datasets:", found_datasets)

""" results = sample_n_from_each_dataset(loaders,n=3)
for ds_name, info in results.items():
    print(f"Dataset: {ds_name}  (file used: {info['file']})  samples: {len(info['samples'])}")
    display(info['df']) """

#print datasets
for name, loader in loaders.items():
    print(f"{name}: {len(loader)} samples")

Found datasets: ['001-zeek-scenario-malicious', '003-zeek-scenario-malicious', '008-zeek-mixed', '009-zeek-malicious', '010-zeek-mixed', '011-zeek-mixed', '012-zeek-mixed', '013-zeek-mixed', '014-zeek-malicious', '015-zeek-malicious', '016-zeek-malicious', '017-zeek-malicious', '018-zeek-malicious', '020-zeek-malicious', '021-zeek-malicious', '022-zeek-malicious', '023-zeek-malicious', '024-zeek-malicious', '025-zeek-malicious', '026-zeek-malicious', '027-zeek-malicious', '028-zeek-malicious', '029-zeek-malicious', '030-zeek-malicious', '031-zeek-malicious', '032-zeek-malicious', '033-zeek-malicious', '034-zeek-malicious', '035-zeek-malicious', '036-zeek-malicious', '037-zeek-mixed']
001-zeek-scenario-malicious: 253 samples
003-zeek-scenario-malicious: 361 samples
008-zeek-mixed: 5603 samples
009-zeek-malicious: 79401 samples
010-zeek-mixed: 5310 samples
011-zeek-mixed: 8722 samples
012-zeek-mixed: 3810 samples
013-zeek-mixed: 26241 samples
014-zeek-malicious: 26738 samples
015-zeek-ma

## Pipeline

In [18]:
#feature processing
feature_extraction = FeatureExtraction()

#preprocessing
scaler = StandardScaler() 
preprocessor = PreprocessingWrapper(experiment_name=experiment_name)
preprocessor.add_step("scaler", scaler)

""" pca = IncrementalPCA(n_components=7)
preprocessor.add_step("pca", pca) """

# other steps here? add your own!

#classifiers:
# sklearn SGD linear
model = SGDClassifier(loss='squared_hinge' , penalty='l2',random_state=seed) 
classifier = SKLearnClassifierWrapper(model,preprocessing_handler=preprocessor)


""" tree_Hoeff = tree.HoeffdingAdaptiveTreeClassifier(seed=seed,max_size=10,max_depth=15)
model = ensemble.ADWINBoostingClassifier(seed=seed,model = tree_Hoeff,n_models=5)


# model = forest.ARFClassifier(seed=seed,n_models=50, max_size=10, warning_detector=drift.ADWIN(delta=0.05))


# final wapper for pipeline
classifier = RiverClassifierWrapper(model,preprocessing_handler=preprocessor)
 """

' tree_Hoeff = tree.HoeffdingAdaptiveTreeClassifier(seed=seed,max_size=10,max_depth=15)\nmodel = ensemble.ADWINBoostingClassifier(seed=seed,model = tree_Hoeff,n_models=5)\n\n\n# model = forest.ARFClassifier(seed=seed,n_models=50, max_size=10, warning_detector=drift.ADWIN(delta=0.05))\n\n\n# final wapper for pipeline\nclassifier = RiverClassifierWrapper(model,preprocessing_handler=preprocessor)\n '

In [19]:
# example of commands to run
# each command is either "train" or "test"
# dataset_prefix is 3 numbers always - which dataset to use (008, 009, 010, ...)
# validation = use validation portion when training

commands = [
    {"command": "train", "dataset_prefix": "001", "validation": False},
    
    {"command":"test", "dataset_prefix":"008"},

]    

In [20]:
# train loop
    # call batch from dataset
    # process features
    # preprocessing (scaling)
    # train on model with validation, logger for metrics!
    # save model after the whole dataset is done
    # reporting, metrics, plots, etc.

rng = np.random.default_rng(seed)

# Ensure log experiment folder exists
experiment_folder = EXPERIMENT_ROOT
if os.path.exists(experiment_folder):
    if os.path.isdir(experiment_folder):
        shutil.rmtree(experiment_folder)
    else:
        os.remove(experiment_folder)
os.makedirs(experiment_folder, exist_ok=False)

# Save config to configs.txt in the experiment folder (for reproducibility)
config_dict = {
    "seed": seed,
    "validation_split": validation_split,
    "commands": commands,
    "experiment_name": experiment_name,
    "root": root
}

# check if the model is fitted (now just to know if we can test from the start or need to train first)
try:
    dummy_input = np.zeros((1, model.n_features_in_))
    classifier.predict(dummy_input)
    is_fitted = True
except Exception:
    print("Model is not fitted.")
    is_fitted = False


# main loop doing commands one by one, and storing logs
for command_idx,command_dict in enumerate(commands):

    #find the dataset we wanted to use
    ds = command_dict["dataset_prefix"]
    try:    
        selected_dataset = next(name for name in found_datasets if ds in name)
        loader = loaders[selected_dataset]
    except StopIteration:
        print(f"No dataset found for {ds}, skipping")
        if command_idx == 0 and not is_fitted:
            print("No dataset for the first training command, exiting")
            exit(1)
        continue

    # based on the command specified, do the action train/test
    command = command_dict["command"]
    num_batches = command_dict.get("batches", None)
    training_batches = loader.batches()
    if num_batches is not None:
        training_batches = min(num_batches, loader.batches())

    if command == "train":
        loader.reset_epoch(batch_size=500)
        logger = Logger(
                logfile_path=logs_dir / f"{command_idx}_{command}_{ds}.log",
                overwrite=True
            )

        print(f"Training on dataset {selected_dataset}")
        do_validation = command_dict.get("validation", False)

        for i in range(training_batches):
            if i %5 == 0:
                print(f"Processing batch {i}")


            batch = loader.next_batch()
            X, y = feature_extraction.process_batch(batch)

            # skip empty batches (no rows or no columns)
            if X is None or getattr(X, "shape", (0, 0))[0] == 0 or getattr(X, "shape", (0, 0))[1] == 0:
                continue

            sum_labeled_flows = len(y)
            if do_validation:
                try:
                    val_size = int(validation_split * X.shape[0])
                    if val_size <= 0 or val_size >= X.shape[0]:
                        # not enough data to split; fall back to training on full batch without validation
                        preprocessor.partial_fit(X)
                        X_processed = preprocessor.transform(X)
                        classifier.partial_fit(X_processed, y)
                        y_pred_train = classifier.predict(X_processed)
                        logger.save_training_results(
                            y_pred_train, y, None, None, sum_labeled_flows
                        )
                        continue

                    validation_indices = rng.choice(
                        X.shape[0],
                        size=val_size,
                        replace=False,
                    )
                    train_indices = np.array(
                        sorted(
                            set(range(X.shape[0])) - set(validation_indices)
                        )
                    )
                    if train_indices.size == 0:
                        continue
                    X_train, X_val = X.iloc[train_indices], X.iloc[validation_indices]
                    y_gt_train, y_gt_val = y.iloc[train_indices], y.iloc[validation_indices]
                except Exception as e:
                    print(f"Error during train_test_split: {e}")
                    continue


                if X_train.shape[0] == 0 or X_train.shape[1] == 0:
                    continue

                #preprocessor
                preprocessor.partial_fit(X_train)
                X_train_processed = preprocessor.transform(X_train)

                #classif
                classifier.partial_fit(X_train_processed, y_gt_train)
                y_pred_train = classifier.predict(X_train_processed)

                # predict on validation set
                if X_val.shape[0] == 0 or X_val.shape[1] == 0:
                    y_pred_val = np.array([])
                else:
                    X_val_processed = preprocessor.transform(X_val)
                    y_pred_val = classifier.predict(X_val_processed)

                logger.save_training_results(
                    y_pred_train, y_gt_train, y_pred_val, y_gt_val, sum_labeled_flows
                )
            else:
                preprocessor.partial_fit(X)
                X_processed = preprocessor.transform(X)
                classifier.partial_fit(X_processed, y)
                y_pred_train = classifier.predict(X_processed)
                logger.save_training_results(
                    y_pred_train, y, None, None, sum_labeled_flows # None is for validation
                )

        # After training, plot the training performance using the external script, not here!


    elif command == "test":
        logger = Logger(
                logfile_path=logs_dir / f"{command_idx}_{command}_{ds}.log",
                overwrite=True
            )
        loader.reset_epoch(batch_size=1_000)
        print(f"Testing on dataset {selected_dataset}")
        for i in range(loader.batches()):
            batch = loader.next_batch()
            if i %25 == 0:
                print(f"Processing batch {i}")
            X, y = feature_extraction.process_batch(batch)
            if X.shape[0] == 0:
                continue
            X_processed = preprocessor.transform(X)
            y_pred = classifier.predict(X_processed)
            logger.save_test_results(y, y_pred)
    else:
        print(f"Unknown command {command}, skipping")
        continue

classifier.save_classifier(path = models_dir ,name = f"model_{experiment_name}.bin")
preprocessor.save(base_path=preproc_dir) 

#  append feature names from the first preprocessing step and model parameters to configs.txt
# get feature names from the first step in the preprocessor
first_preprocessor_step = preprocessor.steps[0][1]
if hasattr(first_preprocessor_step, 'get_feature_names_out'):
    feature_names = first_preprocessor_step.get_feature_names_out()
elif hasattr(first_preprocessor_step, 'feature_names_in_'):
    feature_names = first_preprocessor_step.feature_names_in_
else:
    feature_names = None

# Get model parameters
if hasattr(model, "get_params"):
    model_params = model.get_params()
else:
    model_params = None
model_info = {
    "class": type(model).__name__,
    "loss": getattr(model, "loss", None),
    "params": model_params
}

# Add feature names and model info to config
config_dict["feature_names"] = list(feature_names) if feature_names is not None else None
config_dict["model_info"] = model_info


with open(config_path, "w") as f:
    yaml.safe_dump(config_dict, f)


Model is not fitted.
Training on dataset 001-zeek-scenario-malicious
Processing batch 0
Testing on dataset 008-zeek-mixed
Processing batch 0


In [21]:
# Go through experiment log folders and plot performance for each command

plot_train_script = Path("./plot_utils/plot_train_perf.py").resolve()
plot_test_script = Path("./plot_utils/plot_test_perf.py").resolve()

if not plot_train_script.exists():
    raise FileNotFoundError(f"Plotting script not found: {plot_train_script}")
if not plot_test_script.exists():
    raise FileNotFoundError(f"Plotting script not found: {plot_test_script}")

for idx, cmd in enumerate(commands):

    ds = cmd["dataset_prefix"]
    output_log = f"{idx}_{cmd['command']}_{ds}"

    # logfile produced by Logger
    log_file = Path(experiment_folder) / "logs" / f"{output_log}.log"

    if not log_file.exists():
        print(f"[WARN] Log file not found: {log_file}, skipping plotting")
        continue

    # results_dir is already defined by you and points to plots destination
    results_dir.mkdir(parents=True, exist_ok=True)

    if cmd["command"] == "train":
        subprocess.run(
            [
                "python",
                str(plot_train_script),
                "-f",
                str(log_file),
                "-e",
                output_log,
                "--save_folder",
                str(results_dir),
            ],
            check=True,
        )

    elif cmd["command"] == "test":
        subprocess.run(
            [
                "python",
                str(plot_test_script),
                "-f",
                str(log_file),
                "-e",
                output_log,
                "--save_folder",
                str(results_dir),
            ],
            check=True,
        )


[INFO] Reading logfile: ../experiments/test_test_pipeline/logs/0_train_001.log
[INFO] Accumulating batch and cumulative metrics...
Training plots will be saved to: /home/svobojan/StratosphereLinuxIPS/modules/flowmldetection/pipeline_ml_training/experiments/test_test_pipeline/results/training/0_train_001/training
[INFO] Not enough batches for last-5 (training), skipping.
[INFO] Not enough batches for last-10 (training), skipping.
[INFO] Not enough batches for last-20 (training), skipping.

=== TRAINING Multi-class (Aggregated) ===
Accuracy:             1.0000
Malware F1:           1.0000
Malware FPR:          0.0000
Malware FNR:          0.0000
Macro F1:             1.0000
Precision:            1.0000
Recall:               1.0000
MCC:                  1.0000

=== Per-class metrics (Aggregated) - TRAINING ===
Class                 TP       TN       FP       FN      Acc     Prec      Rec       F1
Benign                 8      244        0        0   1.0000   1.0000   1.0000   1.0000
Malic